#### Imports

In [1]:
import json
import sys

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

sys.path.append("../02-vector-search")
from embedder import Embedder


load_dotenv("../.env")

True

#### Load lesson documents

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

#### Q1 structured output setup

In [3]:
class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


openai_client = OpenAI()

#### Q1 generate questions for first 3 pages

In [4]:
first_three_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

first_three_docs = [
    doc for doc in documents
    if doc["filename"] in first_three_filenames
]

usages = []

for doc in first_three_docs:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.parse(
        model="gpt-5.4-mini",
        input=messages,
        text_format=Questions,
    )

    usages.append(response.usage)

    print()
    print(doc["filename"])
    print(response.output_parsed.questions)
    print(response.usage)


01-agentic-rag/lessons/01-intro.md
['What is a large language model, in simple terms, and how does it generate text?', 'Why does this course treat LLMs like a black box and use an API instead of building one?', 'What problems with LLMs does retrieval help solve, like missing knowledge or made-up answers?', 'What is the main idea behind RAG in this module, and why is it useful for course FAQ answers?', 'What will the first part of the module actually build before the agentic version, and what changes in Part 2?']
ResponseUsage(input_tokens=1021, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=120, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1141)

01-agentic-rag/lessons/02-environment.md
['What do I need installed before I can follow this module, besides knowing some Python and command line basics?', 'How do I set up a brand-new project for this course from an empty folder using uv?', 'Which packages should I

#### Q1 average input tokens

In [5]:
input_tokens = []

for usage in usages:
    input_tokens.append(usage.input_tokens)

average_input_tokens = sum(input_tokens) / len(input_tokens)

average_input_tokens

1354.0

#### Download ground truth

In [6]:
!curl -L -o ground-truth.csv https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 48627  100 48627    0     0   282k      0 --:--:-- --:--:-- --:--:--  282k


#### Load ground truth

In [7]:
ground_truth_df = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth_df.to_dict(orient="records")

len(ground_truth), ground_truth[0]

(360,
 {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'})

#### Create chunks

In [8]:
chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

#### Build text search

In [9]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(chunks)


def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

#### Build vector search

In [10]:
embedder = Embedder(path="../02-vector-search/models/Xenova/all-MiniLM-L6-v2")

texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(texts)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)


def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vindex.search(query_vector, num_results=num_results)

#### Build hybrid search

In [11]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)

    return rrf([text_results, vector_results], k=k)

#### Q2 and Q3

In [12]:
q = ground_truth[0]["question"]

text_results = text_search(q)
vector_results = vector_search(q)

print("Question:", q)

print("Q2 text search first result:")
print(text_results[0]["filename"])

print()

print("Q3 vector search first result:")
print(vector_results[0]["filename"])

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q2 text search first result:
01-agentic-rag/lessons/03-rag.md

Q3 vector search first result:
01-agentic-rag/lessons/01-intro.md


#### Evaluation functions

In [13]:
def compute_relevance(record, search_function):
    query = record["question"]
    expected_filename = record["filename"]

    results = search_function(query)

    relevance = []

    for result in results:
        relevance.append(int(result["filename"] == expected_filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for record in ground_truth:
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance_total)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(
        ground_truth,
        search_function,
    )

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

#### text search evaluation

In [14]:
text_eval = evaluate(ground_truth, text_search)

text_eval

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

#### vector search evaluation

In [15]:
vector_eval = evaluate(ground_truth, vector_search)

vector_eval

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

#### hybrid tuning

In [16]:
for k in [1, 50, 100, 200]:
    def hybrid_search_k(query, k=k):
        return hybrid_search(query, k=k)

    result = evaluate(ground_truth, hybrid_search_k)

    print("k:", k)
    print(result)
    print()

k: 1
{'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}

k: 50
{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

k: 100
{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

k: 200
{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

